In [22]:
##################################################################
# CONNECT TO BLACKBLAZE
from babel.dates import format_datetime
import pytz
from datetime import datetime
from b2sdk.v2 import InMemoryAccountInfo, B2Api
import logging

##################################################################
# LOG MESSAGES ON CONSOLE AND FILE
def log_msg(msg):
    logging.info(msg)
    d = datetime.now(pytz.timezone('Europe/Paris'))
    d_str = format_datetime(d, "d-MM-y HH:mm:ss", locale='fr_FR')
    if not isinstance(msg, str):
        msg = str(msg)
    print(d_str + ' ' + msg,flush=True)

def get_b2_bucket(bucket_name):
    b2_info = InMemoryAccountInfo()
    b2_api = B2Api(b2_info)
    if bucket_name == 'meloir':
        b2_application_key_id = os.getenv("B2_MELOIR_KEY_ID")
        b2_application_key = os.getenv("B2_MELOIR_APPLICATION_KEY")
        b2_name = 'MeloirFiles'
    elif bucket_name == 'temperature':
        b2_application_key_id = os.getenv("B2_MELOIR_KEY_ID")
        b2_application_key = os.getenv("B2_MELOIR_APPLICATION_KEY")
        b2_name = 'MeloirFiles'
    elif bucket_name == 'berger':
        b2_application_key_id = os.getenv('B2_BERGER_KEY_ID')
        b2_application_key = os.getenv('B2_BERGER_APPLICATION_KEY')
        b2_name = 'bergerbookings'
    elif bucket_name == 'bikedata':
        b2_application_key_id = os.getenv('B2_BIKEDATA_KEY_ID')
        b2_application_key = os.getenv('B2_BIKEDATA_APPLICATION_KEY')
        b2_name = 'bikedata'
    elif bucket_name == 'bergerconfessions':
        b2_application_key_id = os.getenv('B2_BERGERCONFESSIONS_KEY_ID')
        b2_application_key = os.getenv('B2_BERGERCONFESSIONS_APPLICATION_KEY')
        b2_name = 'bergerconfessions'
    elif bucket_name == 'bergermesses':
        b2_application_key_id = os.getenv('B2_BERGERMESSES_KEY_ID')
        b2_application_key = os.getenv('B2_BERGERMESSES_APPLICATION_KEY')
        b2_name = 'bergermesses'
    elif bucket_name == 'bergershops':
        b2_application_key_id = os.getenv('B2_BERGERSHOPS_KEYID')
        b2_application_key = os.getenv('B2_BERGERSHOPS_APPLICATION_KEY')
        b2_name = 'bergershops'
    else:
        raise ValueError("Unknown bucket name: " + str(bucket_name))
    b2_api.authorize_account("production", b2_application_key_id, b2_application_key)
    
    return b2_api.get_bucket_by_name(b2_name)

##################################################################
# UPLOAD FILE TO BLACKBLAZE
def push_b2_file(bucket_name, file_local, file_BB):
    bucket = get_b2_bucket(bucket_name)
    bucket.upload_local_file(
        local_file=file_local,
        file_name=file_BB
    )
##################################################################
# DOWNLOAD FILE FROM BLACKBLAZE
def download_file_from_b2(bucket_name, file_name_BB, local_path):
    log_msg('Getting bucket...')
    bucket = get_b2_bucket(bucket_name)
    log_msg('Done')
    log_msg('Downloading file from BB:'+ str(file_name_BB))
    x = bucket.download_file_by_name(file_name_BB)
    log_msg('Save file '+ str(local_path))
    x.save_to(local_path)
    log_msg('Done')
    log_msg(f"Downloaded '{file_name_BB}' to '{local_path}'")


In [26]:
import os, time, json
import requests, re
from typing import Tuple, Dict, Any, Optional, List
from openai import OpenAI
import pandas as pd
import re, html
from datetime import date, timedelta, datetime
import unicodedata
import urllib.parse
import unicodedata

##################################################################
# INITIALISATION

#root_path = '/Users/etiennecomon/Downloads/cinema/'
TIME_RE = re.compile(r"\b\d{1,2}[:h]\d{2}\s*(?:[ap]m)?\b", re.I)
PICKLE_PARIS_FILMS_BB = 'berger_films.pickle'
PICKLE_PARIS_FAILED_FILMS_BB = 'berger_films_failed.pickle'
PERPLEXITY_MODEL = "sonar-pro" #"llama-3.1-sonar-large-128k-online"
TIME_RE = re.compile(r"\b\d{1,2}[:h]\d{2}\s*(?:[ap]m)?\b", re.I)

# list of cinemas (all Paris)
list_cinemas = ["Grand Action", "Reflet Médicis", "Espace Saint Michel", "Epée de Bois", "Champo", "Filmothèque du Quartier Latin","Cinéma Christine","Ecoles Cinéma Club","Cinéma du Panthéon","Studio Galande", "Studio des Ursulines", "Cinéma Arlequin", "Cinéma Lincoln","Cinéma Balzac","Cinéma des Cinéastes", "Sept Parnassiens"]
cinema_location = 'Paris, France'
root_path = '/Users/etiennecomon/Downloads/cinema/'

# List of spurious film names
list_film_entries_ignore = ['Christine 21','Christine Cinéma','Cinéma des cin','Cinéma du Panth','Cinéma Espace Saint','Cinéma Studio','Espace Saint-Michel','Espace Saint Michel','GRAND ACTION','Le Balzac','Les horaires du cin','Studio Galande','À L&#x27;AFFICHE','Écoles Cinéma','Cinéma La Filmot','Reflet Medicis','Reflet Médicis','Cinéma le Lincoln','Filmothèque du Quar','Cinéma Epée de Bois','Programme TV','Espace Marcel','horaires des films','La Filmothèque du Quarti']

list_film_entries_ignore = [f.lower() for f in list_film_entries_ignore]

In [6]:
##################################################################
# FUNCTIONS SCRAPING FILM TIMES FROM GOOGLE VIA SERP API

# One cinema - get show listings from Google via Serp API
def get_showtimes_serpapi_one_cinema(cinema: str, location: Optional[str] = None, timeout: int = 30) -> Tuple[Dict[str, Any], Dict[str, list]]:
    api_key = os.getenv('SERP_API')
    q = f"showtimes {cinema} {location or ''}".strip()
    params = {'engine': 'google', 'q': q, 'hl': 'en', 'api_key': api_key}
    if location:
        params['location'] = location
    r = requests.get('https://serpapi.com/search.json', params=params, timeout=timeout)
    r.raise_for_status()
    data = r.json()
    matches: dict[str, list] = {}
    def traverse(obj, path='root'):
        if isinstance(obj, str):
            found = TIME_RE.findall(obj)
            if found:
                matches[path] = found
        elif isinstance(obj, dict):
            for k, v in obj.items():
                traverse(v, f"{path}/{k}")
        elif isinstance(obj, list):
            for i, v in enumerate(obj):
                traverse(v, f"{path}[{i}]")
    traverse(data, 'root')
    return data, matches

# All cinemas - get show listings from Google via Serp API
def get_showtimes_serpapi_all_cinemas(list_cinemas):
    all_data, all_matches = {}, {}
    for c in list_cinemas:
        try:
            log_msg('Querying SerpAPI for ' + str(c))
            d, m = get_showtimes_serpapi_one_cinema(c, cinema_location)
            all_data[c] = d
            all_matches[c] = m
            time.sleep(1)
        except Exception as e:
            log_msg('Failed for ' + str(c) + ' | m' + str(e))
    return all_data, all_matches





In [7]:

##################################################################
# FUNCTIONS CLEANING AND ORGANISING FILM TIME SCRAPE DATA

# Helper to sanitize filenames
def _sanitize_fname(s: str) -> str:
    if not s:
        return 'untitled'
    s = unicodedata.normalize('NFKD', s)
    s = s.encode('ascii', 'ignore').decode('ascii')
    s = re.sub(r'[^A-Za-z0-9._-]+', '_', s)
    return s[:120]

# Convert text label to ISO date where possible
def to_iso_date_from_label(label: str):
    if not label or not isinstance(label, str):
        return None
    s_raw = label.strip()
    month_map = {
        'janvier':'January','février':'February','fevrier':'February','mars':'March','avril':'April','mai':'May','juin':'June',
        'juillet':'July','août':'August','aout':'August','septembre':'September','octobre':'October','novembre':'November','décembre':'December','decembre':'December',
        'janv':'Jan','fév':'Feb','fev':'Feb','avr':'Apr','juil':'Jul','août':'Aug','aout':'Aug','sept':'Sep','oct':'Oct','nov':'Nov','déc':'Dec','dec':'Dec'
    }
    s_norm = s_raw
    for fr, en in month_map.items():
        s_norm = re.sub(fr, en, s_norm, flags=re.I)
    s = s_norm.lower()
    today = date.today()
    if 'today' in s or 'aujourd' in s:
        return today.isoformat()
    if 'tomorrow' in s or 'demain' in s:
        return (today + timedelta(days=1)).isoformat()
    m = re.search(r'\b(\d{4}-\d{2}-\d{2})\b', s)
    if m:
        return m.group(1)
    m2 = re.search(r'\b(\d{1,2}[/-]\d{1,2}[/-]\d{2,4})\b', s)
    if m2:
        for fmt in ('%d/%m/%Y','%d-%m-%Y','%d/%m/%y','%d-%m-%y'):
            try:
                dt = datetime.strptime(m2.group(1), fmt).date()
                return dt.isoformat()
            except Exception:
                pass
    month_formats = ['%d %B %Y', '%d %b %Y', '%B %d %Y', '%b %d %Y', '%d %B', '%d %b', '%B %d', '%b %d']
    for fmt in month_formats:
        try:
            dt = datetime.strptime(s_raw, fmt).date()
            if dt.year == 1900:
                dt = dt.replace(year=today.year)
                if dt < today:
                    dt = dt.replace(year=today.year + 1)
            return dt.isoformat()
        except Exception:
            pass
    weekdays = {'monday':0,'tuesday':1,'wednesday':2,'thursday':3,'friday':4,'saturday':5,'sunday':6,
                'lundi':0,'mardi':1,'mercredi':2,'jeudi':3,'vendredi':4,'samedi':5,'dimanche':6}
    for name, wd in weekdays.items():
        if name in s:
            days_ahead = (wd - today.weekday() + 7) % 7
            target = today if days_ahead == 0 else today + timedelta(days=days_ahead)
            return target.isoformat()
    return None

# Collect time strings from nested SerpAPI structures
def collect_times(node):
    times = []
    if isinstance(node, str):
        times.extend(TIME_RE.findall(node))
    elif isinstance(node, dict):
        for v in node.values():
            times.extend(collect_times(v))
    elif isinstance(node, list):
        for v in node:
            times.extend(collect_times(v))
    return list(dict.fromkeys([t.strip() for t in times if t]))

# Walk SerpAPI response to extract film entries while preserving day context
def extract_from_cinema(data, cinema_name):
    films = []
    def walk(node, context_day=None):
        if isinstance(node, list):
            if all(isinstance(el, dict) for el in node) and any(('day' in el or 'date' in el or 'movies' in el) for el in node):
                for el in node:
                    raw_day = ''
                    if isinstance(el, dict):
                        raw_day = (el.get('date') or el.get('day') or el.get('title') or '')
                    iso_day = to_iso_date_from_label(raw_day) or (raw_day if raw_day else 'unknown')
                    movies_list = el.get('movies') if isinstance(el, dict) else None
                    if movies_list and isinstance(movies_list, list):
                        for m in movies_list:
                            title = m.get('name') or m.get('title') or m.get('movie')
                            times = collect_times(m)
                            if title and times:
                                films.append({'title': title, 'cinema': cinema_name, 'day': iso_day, 'times': times, 'meta': m})
                        continue
                    walk(el, context_day=iso_day)
                return
            for el in node:
                walk(el, context_day=context_day)
        elif isinstance(node, dict):
            if 'movies' in node and isinstance(node.get('movies'), list):
                raw_day = (node.get('date') or node.get('day') or node.get('title') or '')
                iso_day = to_iso_date_from_label(raw_day) or (raw_day if raw_day else context_day or 'unknown')
                for m in node.get('movies', []):
                    title = m.get('name') or m.get('title') or m.get('movie')
                    times = collect_times(m)
                    if title and times:
                        films.append({'title': title, 'cinema': cinema_name, 'day': iso_day, 'times': times, 'meta': m})
                return
            if any(k.lower() in ('title','name','movie') for k in node.keys()):
                title = node.get('title') or node.get('name') or node.get('movie')
                times = collect_times(node)
                if title and times:
                    films.append({'title': title, 'cinema': cinema_name, 'day': context_day or 'unknown', 'times': times, 'meta': node})
                return
            for v in node.values():
                walk(v, context_day=context_day)
    walk(data)
    return films



# Aggregate films across cinemas — NO director/year/country extraction here
def aggregate_films_across_cinemas(data_all, matches_all):
    agg: Dict[str, Dict[str, Any]] = {}
    for cinema, d in data_all.items():
        try:
            films = extract_from_cinema(d, cinema)
        except Exception as e:
            print('Extraction error for', cinema, e)
            films = []
        for f in films:
            title_norm = (f.get('title') or '').strip()
            if not title_norm:
                continue
            # drop pure time-like titles
            try:
                if TIME_RE.fullmatch(title_norm):
                    continue
            except Exception:
                pass
            key = title_norm.lower()
            entry = agg.setdefault(key, {'title': title_norm, 'cinemas': set(), 'schedule': {}, 'meta_sample': None, 'google_link': ''})
            entry['cinemas'].add(f['cinema'])
            day_iso = f['day']
            if day_iso == 'unknown':
                meta = f.get('meta') or {}
                for v in meta.values() if isinstance(meta, dict) else []:
                    if isinstance(v, str):
                        iso = to_iso_date_from_label(v)
                        if iso:
                            day_iso = iso
                            break
            resolved = to_iso_date_from_label(day_iso) if isinstance(day_iso, str) else None
            if not resolved:
                resolved = 'unknown'
            sched = entry['schedule'].setdefault(resolved, set())
            for t in f['times']:
                sched.add(t)
            try:
                if not entry.get('meta_sample') and f.get('meta') is not None:
                    entry['meta_sample'] = f.get('meta')
            except Exception:
                pass
            # store a Google search link for the title (for later OpenAI use)
            if not entry.get('google_link'):
                q = urllib.parse.quote_plus(title_norm)
                entry['google_link'] = f"https://www.google.com/search?hl=fr&gl=FR&q={q}"
    return agg


# Build table with film times
def make_table_film_times(agg, list_films_ignore):
    rows = []
    list_removed = []
    for k, v in sorted(agg.items(), key=lambda x: x[1]['title'].lower()):
        cinemas = ', '.join(sorted(v['cinemas']))
        day_parts = []
        for day in sorted(v['schedule'].keys()):
            times = ', '.join(sorted(v['schedule'][day]))
            if day == 'unknown':
                label = 'unknown'
            else:
                try:
                    d_obj = datetime.fromisoformat(day)
                    label = d_obj.strftime('%a %d-%b')
                except Exception:
                    label = day
            day_parts.append(f"{label}: {times}")
        title_text = v['title']
        google_url = v.get('google_link','')
        if any(f_ignore in title_text.lower() for f_ignore in list_films_ignore):
            list_removed.append(title_text[:15])
        else:
            rows.append({'Title': title_text, 'Film link': google_url, 'Cinemas': cinemas, 'Schedule': '\n'.join(day_parts)})
    table_films = pd.DataFrame(rows)
    if len(list_removed) > 0:
        log_msg('Spurious film titles skipped: ' + ' | '.join(list_removed))

    return table_films


In [ ]:
if True:
    log_msg('Fetching film times of individual cinemas')
    all_data, all_matches = get_showtimes_serpapi_all_cinemas(list_cinemas)
    log_msg('Aggregating film times across cinemas')
    x_all_films = aggregate_films_across_cinemas(all_data, all_matches)
log_msg('Making films table')
table_films = make_table_film_times(x_all_films, list_film_entries_ignore)
log_msg('Done films table')

31-12-2025 16:48:58 Making films table
31-12-2025 16:48:58 Spurious film titles skipped: Christine 21 -  | Christine Ciném | Cinéma des cine | Cinéma des Ciné | Cinéma du Panth | Cinéma Epée de  | Cinéma Studio G | Cinéma Studio-G | Espace Marcel C | Espace Saint-Mi | Le Balzac | Le Cinéma des c | Le Cinéma des C | Le Grand Action | Les horaires de | Les horaires du | Programme TV Ci | Studio Galande  | Écoles Cinéma C | Écoles Cinéma C | Écoles Cinéma C | Écoles Cinéma C
31-12-2025 16:48:58 Done films table


In [ ]:
# Perplexity setup
PERPLEXITY_MAX_FILMS = 300
PERPLEXITY_MAX_TRY = 3
PERPLEXITY_TIMEOUT_MIN = 4
pp_api_key = os.getenv("PERPLEXITY_KEY")
perplexity_client = OpenAI(api_key=pp_api_key, base_url="https://api.perplexity.ai")

######################################################################
# GET DIRECTORY, FILM YEAR, COUNTRY, GENRE, SYNOPSIS FROM PERPLEXITY

# Get from BlackBlaze the existing set of film references
def get_existing_films():
    download_file_from_b2('bergershops', PICKLE_PARIS_FILMS_BB, PICKLE_PARIS_FILMS_BB)
    with open(PICKLE_PARIS_FILMS_BB, 'rb') as f:
        x_existing = pd.read_pickle(f)
    download_file_from_b2('bergershops', PICKLE_PARIS_FAILED_FILMS_BB, PICKLE_PARIS_FAILED_FILMS_BB)
    with open(PICKLE_PARIS_FAILED_FILMS_BB, 'rb') as f:
        x_failed = pd.read_pickle(f)
    return x_existing, x_failed

# Query film descriptions from Perplexity
def query_films_from_perplexity(pp_client, film_name, film_link, msg_tracker, failed_storage):
    log_msg('Perplexity query [' + msg_tracker + '] for ' + film_name)
    prompt = ("I will provide a film with a google link. Please return a completed dictionary where you will add the following elements: [a] 'Director', [b] 'Year', [c] 'Country',[d] 'Genre',[e] 'Synopsis' based on the film name and google link provided. " +
            "Return the completed dictionary for this film. " +
            "If a value is not present, return an empty string for that field. " +
            "Make the response a pure json string, without any comment, prefix nor suffix. Now the dictionary:\n\n" + json.dumps({'Film':film_name, 'link':film_link}))
    resp = pp_client.chat.completions.create(
        model=PERPLEXITY_MODEL,
        messages=[
            {"role":"user","content":prompt}
        ],
        max_tokens=128000,
        temperature=0.0,
        timeout=60*PERPLEXITY_TIMEOUT_MIN
    )
    if not resp.choices:
        raise RuntimeError("No response from Perplexity API")
    if resp.choices[0].finish_reason != 'stop':
        raise RuntimeError("Incomplete response from Perplexity API")
    s = resp.choices[0].message.content
    if s.count('{') != s.count('}'):
        raise RuntimeError("Mismatched { and }")
    if "```" in s:
        s = s[s.find("```")+3 : s.rfind('```')]
    s = s.replace('json\n','')

    # Parse JSON content and store into pandas
    resp_converted = json.loads(s)

    # Convert to Pandas
    resp_df = pd.DataFrame([resp_converted])
    return resp_df

# Set up the list of films to query from Perplexity
x_existing_perplexity, x_failed = get_existing_films()
prompt_films = {}
count_new_film = 0
for ix in table_films.index[:PERPLEXITY_MAX_FILMS]:
    if (not table_films.loc[ix, 'Title'] in x_existing_perplexity.index):
        prompt_films[count_new_film] = {'Title': table_films.loc[ix,'Title'], 'Film link': table_films.loc[ix,'Film link']}
        count_new_film += 1

print('n prompts ', len(prompt_films))
# Loop through batches of films
list_new_perplexity = []
list_fails = []
for i_film in range(0, len(prompt_films)):
    # Define the batch of films to query
    str_tracker = str(i_film+1) + '/' + str(len(prompt_films))

    # Run the Perplexity query until successful
    n_tries = 0
    completed = False
    while not completed and n_tries < PERPLEXITY_MAX_TRY:
        film_name = prompt_films[i_film]['Title']
        film_link = prompt_films[i_film]['Film link']
        try:
            response = query_films_from_perplexity(perplexity_client, film_name,film_link, str_tracker, x_failed)
            if response[(~pd.isnull(response['Director'])&(response['Director']!=''))].shape[0] == 1:
                completed = True
            else:
                n_tries += 1
                log_msg(f"Incomplete data in film {film_name}, try {n_tries}")
                time.sleep(2)
                response = None
        except Exception as e:
            n_tries += 1
            log_msg(f"Error querying film {film_name}, try {n_tries}: {e}")
            time.sleep(2)
            response = None
    if not response is None:    
        list_new_perplexity.append(response)
    else:
        list_fails.append(film_name)
        if film_name in x_failed.index:
            x_failed.loc[film_name,'n_fail'] = x_failed.loc[film_name,'n_fail'] + 1
        else:
            x_failed.loc[film_name,'n_fail'] = 1
# Concatenate new film entries
if len(list_new_perplexity) > 0:
    x_new_perplexity = pd.concat(list_new_perplexity)
    x_new_perplexity.set_index('Film',inplace=True)
    x_new_perplexity = x_new_perplexity[~pd.isnull(x_new_perplexity['Director'])]
    x_new_perplexity = x_new_perplexity[x_new_perplexity['Director'] != '']
    if 'Link' in x_new_perplexity.columns:
        x_new_perplexity.drop(columns=['Link'],inplace=True)

    # Combine old and new film entries from Perplexity
    log_msg('Combining existing and new film Perplexity entries')
    log_msg('\tExisting = %d' % x_existing_perplexity.shape[0])
    log_msg('\tNew = %d' % x_new_perplexity.shape[0])
    if x_new_perplexity.shape[0] > 0:
        x_perplexity = pd.concat([x_existing_perplexity, x_new_perplexity])
        x_perplexity = x_perplexity[~x_perplexity.index.duplicated(keep='first')]
    else:
        x_perplexity = x_existing_perplexity.copy()
else:
    x_perplexity = x_existing_perplexity.copy()

log_msg('\tCombined = %d' % x_perplexity.shape[0])
x_perplexity.to_clipboard()

# Save result
log_msg('Saving combined Perplexity film entries to BlackBlaze')
x_perplexity.to_pickle(PICKLE_PARIS_FILMS_BB)
push_b2_file('bergershops', PICKLE_PARIS_FILMS_BB, PICKLE_PARIS_FILMS_BB)
log_msg('Done saving combined Perplexity film entries to BlackBlaze')

# Show failed films
if len(list_fails) > 0:
    log_msg('Failed films:' + '\n'.join(['\t'+s for s in list_fails]))


31-12-2025 16:49:00 Getting bucket...
31-12-2025 16:49:02 Done
31-12-2025 16:49:02 Downloading file from BB:berger_films.pickle
31-12-2025 16:49:03 Save file berger_films.pickle
31-12-2025 16:49:03 Done
31-12-2025 16:49:03 Downloaded 'berger_films.pickle' to 'berger_films.pickle'
31-12-2025 16:49:03 Getting bucket...
31-12-2025 16:49:05 Done
31-12-2025 16:49:05 Downloading file from BB:berger_films_failed.pickle
31-12-2025 16:49:05 Save file berger_films_failed.pickle
31-12-2025 16:49:05 Done
31-12-2025 16:49:05 Downloaded 'berger_films_failed.pickle' to 'berger_films_failed.pickle'
n prompts  1
31-12-2025 16:49:05 Perplexity query [1/1] for Mektoub, My love: Canto Uno
31-12-2025 16:49:09 Combining existing and new film Perplexity entries
31-12-2025 16:49:09 	Existing = 300
31-12-2025 16:49:09 	New = 1
31-12-2025 16:49:09 	Combined = 301
31-12-2025 16:49:09 Saving combined Perplexity film entries to BlackBlaze
31-12-2025 16:49:14 Done saving combined Perplexity film entries to BlackBla

In [22]:
######################################################################
# GET TRAILER LINK AND WIKI PAGE FOR EACH FILM

x_perplexity, x_failed = get_existing_films()
x_perplexity['trailer_url'] = None
x_perplexity['wikipedia_title'] = None
perplexity_client = OpenAI(api_key=pp_api_key, base_url="https://api.perplexity.ai")

for film_name in x_perplexity.index:
    if isinstance(film_name, str):
        if pd.isnull(x_perplexity.loc[film_name, 'Trailer link']) or pd.isnull(x_perplexity.loc[film_name, 'Wikipedia']):
            print(film_name, ' trailer=', x_perplexity.loc[film_name, 'trailer_url'], ' wiki=', x_perplexity.loc[film_name, 'wikipedia_title'])
            prompt = "Please provide [a] a YouTube link to the movie trailer if available; and [b] a link to a jpeg or png picture of the movie poster for the film if available on Wikipedia. The film is '" + film_name + "' by " + film_director + ". Please provide your answer as a json dictionary of two elements, with no commentary or addition. If either [a] or [b] cannot be returned, please return an empty string for the missing item"
            prompt = f"""You are a web-connected assistant.
                        Task:
                        For the film "{film_name}" directed by {film_director}:

                        1. Find an official or widely used movie trailer on YouTube.
                        - Return the full YouTube URL as "trailer_url".

                        2. The exact English Wikipedia article title of the movie (case sensitive, exactly as used in the page URL, without adding https:// links). 

                        Output format (IMPORTANT):
                        - Return ONLY a single JSON object (no commentary, no extra text).
                        - The JSON must have exactly these keys:
                        - "trailer_url": string or null
                        - "wikipedia_title": string or null

                        If you truly cannot find a valid   URL, set the field to null, but do not omit the key.

                        Example of the required structure (for illustration only):

                        {{
                        'trailer_url': 'https://www.youtube.com/...',
                        'wikipedia_title": "Amores_Perros'
                        }}
                        """

            messages = [
                {
                    "role": "system",
                    "content": (
                        "You find official movie trailer and poster URLs. "
                        "Return ONLY valid JSON dictionary with the following keys: "
                        "trailer_url, poster_url. Do not include explanations."
                    ),
                },
                {
                    "role": "user",
                    "content": prompt
                }
            ]
            n_tries = 0
            completed = False
            while not completed and n_tries < PERPLEXITY_MAX_TRY:
                try:
                    resp = perplexity_client.chat.completions.create(
                        model=PERPLEXITY_MODEL,
                        messages=messages,
                        max_tokens=128000,
                        temperature=0.4,
                        timeout=60*PERPLEXITY_TIMEOUT_MIN
                    )
                    try:
                        resp = resp.choices[0].message.content
                        resp = resp.replace("```","")
                        try:
                            resp_converted = json.loads(resp)
                            print(resp_converted)
                            x_perplexity.loc[film_name, 'Trailer link'] = resp_converted.get('trailer_url','')
                            x_perplexity.loc[film_name, 'Wikipedia'] = resp_converted.get('wikipedia_title','')
                            completed = True
                        except Exception as e:
                            print('Could not convert results for ' + film_name + ' | '  + str(e))
                            n_tries += 1
                    except Exception as e:
                        print('No output for ' + film_name + '[' + str(e) + ']')
                        n_tries += 1
                except Exception as e:
                    print('Perplexity failed for ' + film_name + '[' + str(e) + ']')
                    n_tries += 1

# Save result
log_msg('Saving Perplexity trailers and Wiki pages to BlackBlaze')
x_perplexity.to_pickle(PICKLE_PARIS_FILMS_BB)
push_b2_file('bergershops', PICKLE_PARIS_FILMS_BB, PICKLE_PARIS_FILMS_BB)
log_msg('Done saving Perplexity trailers and Wiki pages to BlackBlaze')

31-12-2025 17:21:39 Getting bucket...
31-12-2025 17:21:41 Done
31-12-2025 17:21:41 Downloading file from BB:berger_films.pickle
31-12-2025 17:21:41 Save file berger_films.pickle
31-12-2025 17:21:42 Done
31-12-2025 17:21:42 Downloaded 'berger_films.pickle' to 'berger_films.pickle'
31-12-2025 17:21:42 Getting bucket...
31-12-2025 17:21:44 Done
31-12-2025 17:21:44 Downloading file from BB:berger_films_failed.pickle
31-12-2025 17:21:44 Save file berger_films_failed.pickle
31-12-2025 17:21:44 Done
31-12-2025 17:21:44 Downloaded 'berger_films_failed.pickle' to 'berger_films_failed.pickle'
Confessions of a Cheat  trailer= None  wiki= None
Could not convert results for Confessions of a Cheat | Expecting value: line 1 column 1 (char 0)
Could not convert results for Confessions of a Cheat | Expecting value: line 1 column 1 (char 0)
Could not convert results for Confessions of a Cheat | Expecting value: line 1 column 1 (char 0)
La Malibran  trailer= None  wiki= None
{'trailer_url': None, 'wikiped

In [23]:
x_perplexity.to_clipboard()

In [ ]:
############################################################
# ADD WIKIPEDIA LINK

from urllib.parse import quote

def wikipedia_page_url(title: str) -> str:
    normalized = title.replace(" ", "_")
    return "https://en.wikipedia.org/wiki/" + quote(normalized)

log_msg('Loading films')
x_perplexity, x_failed = get_existing_films()
if not 'Wikipedia page' in x_perplexity.columns:
    x_perplexity['Wikipedia page'] = None

log_msg('Adding Wikipedia page links')
for f in x_perplexity.index:
    if pd.isnull(x_perplexity.loc[f,'Wikipedia page']):
        if not pd.isnull(x_perplexity.loc[f,'Wikipedia']):
            x_perplexity.loc[f,'Wikipedia page'] = wikipedia_page_url(x_perplexity.loc[f,'Wikipedia'])


log_msg('Saving Wikipedia page links')
x_perplexity.to_pickle(PICKLE_PARIS_FILMS_BB)
push_b2_file('bergershops', PICKLE_PARIS_FILMS_BB, PICKLE_PARIS_FILMS_BB)
log_msg('Done saving Wikipedia page links into BlackBlaze')

31-12-2025 17:26:26 Loading films
31-12-2025 17:26:26 Getting bucket...
31-12-2025 17:26:28 Done
31-12-2025 17:26:28 Downloading file from BB:berger_films.pickle
31-12-2025 17:26:29 Save file berger_films.pickle
31-12-2025 17:26:29 Done
31-12-2025 17:26:29 Downloaded 'berger_films.pickle' to 'berger_films.pickle'
31-12-2025 17:26:29 Getting bucket...
31-12-2025 17:26:31 Done
31-12-2025 17:26:31 Downloading file from BB:berger_films_failed.pickle
31-12-2025 17:26:32 Save file berger_films_failed.pickle
31-12-2025 17:26:32 Done
31-12-2025 17:26:32 Downloaded 'berger_films_failed.pickle' to 'berger_films_failed.pickle'
31-12-2025 17:26:32 Adding Wikipedia page links
31-12-2025 17:26:32 Saving Wikipedia page links
31-12-2025 17:26:35 Done saving Wikipedia page links into BlackBlaze


In [25]:
############################################################
# ADD FILM POSTERS

import requests

log_msg('Loading films')
x_perplexity, x_failed = get_existing_films()

HEADERS = {
    # Use your own app name + contact info here
    "User-Agent": "FilmPosterFetcher/1.0 (your_email@example.com)"
}

def _query_wikipedia(params: dict) -> dict:
    api = "https://en.wikipedia.org/w/api.php"
    r = requests.get(api, params=params, headers=HEADERS, timeout=10)
    r.raise_for_status()
    return r.json()

def _get_page_dict_for_title(page_title: str) -> dict | None:
    data = _query_wikipedia({
        "action": "query",
        "format": "json",
        "titles": page_title,
        "redirects": 1,
    })
    pages = data.get("query", {}).get("pages", {})
    if not pages:
        return None
    return next(iter(pages.values()))

def get_wikipedia_poster_url(page_title: str, thumb_width: int = 300) -> str | None:
    """
    Try to get a poster-like image URL for a given English Wikipedia page.
    1) First try pageimages thumbnail.
    2) Then fall back to images + imageinfo and pick a 'poster' file.
    Returns a direct jpg/png URL or None.
    """

    # --- 1) Try pageimages thumbnail ---
    data = _query_wikipedia({
        "action": "query",
        "format": "json",
        "prop": "pageimages",
        "piprop": "thumbnail",
        "pithumbsize": thumb_width,
        "titles": page_title,
        "redirects": 1,
    })

    pages = data.get("query", {}).get("pages", {})
    if pages:
        page = next(iter(pages.values()))
        thumb = page.get("thumbnail")
        if thumb and "source" in thumb:
            return thumb["source"]

    # Optional: if that failed, try with " (film)" suffix
    alt_title = f"{page_title} (film)"
    if alt_title != page_title:
        data_alt = _query_wikipedia({
            "action": "query",
            "format": "json",
            "prop": "pageimages",
            "piprop": "thumbnail",
            "pithumbsize": thumb_width,
            "titles": alt_title,
            "redirects": 1,
        })
        pages_alt = data_alt.get("query", {}).get("pages", {})
        if pages_alt:
            page_alt = next(iter(pages_alt.values()))
            thumb_alt = page_alt.get("thumbnail")
            if thumb_alt and "source" in thumb_alt:
                return thumb_alt["source"]

    # --- 2) Fallback: list images on the page and pick a poster-like one ---
    page = _get_page_dict_for_title(page_title)
    if page is None or "pageid" not in page:
        return None

    pageid = page["pageid"]
    images_data = _query_wikipedia({
        "action": "query",
        "format": "json",
        "prop": "images",
        "pageids": pageid,
        "imlimit": "max",
    })

    img_page = images_data.get("query", {}).get("pages", {}).get(str(pageid), {})
    images = img_page.get("images", [])
    if not images:
        return None

    def is_image_file(title: str) -> bool:
        t = title.lower()
        return t.endswith(".jpg") or t.endswith(".jpeg") or t.endswith(".png")

    # Prefer filenames that look like posters
    poster_candidates = [
        img["title"] for img in images
        if is_image_file(img.get("title", ""))
        and any(
            token in img["title"].lower()
            for token in ("poster", "film_poster", "movie_poster")
        )
    ]

    # If none look like posters, fall back to any jpg/png
    if not poster_candidates:
        poster_candidates = [
            img["title"] for img in images
            if is_image_file(img.get("title", ""))
        ]

    if not poster_candidates:
        return None

    image_title = poster_candidates[0]

    # --- 3) Get direct URL from imageinfo ---
    imageinfo_data = _query_wikipedia({
        "action": "query",
        "format": "json",
        "titles": image_title,
        "prop": "imageinfo",
        "iiprop": "url",
    })

    img_pages = imageinfo_data.get("query", {}).get("pages", {})
    if not img_pages:
        return None

    img_page = next(iter(img_pages.values()))
    imageinfo = img_page.get("imageinfo")
    if not imageinfo:
        return None

    return imageinfo[0].get("url")


# Loop through films
log_msg('Adding film posters')
if 'Poster' not in x_perplexity.columns:
    x_perplexity['Poster'] = None
for f in x_perplexity.index:
    if not x_perplexity.loc[f, 'Wikipedia'] is None:
        if pd.isnull(x_perplexity.loc[f,'Poster']):
            picture_url = get_wikipedia_poster_url(x_perplexity.loc[f,'Wikipedia'])
            x_perplexity.loc[f,'Poster'] = picture_url
log_msg('Saving poster links')
x_perplexity.to_pickle(PICKLE_PARIS_FILMS_BB)
push_b2_file('bergershops', PICKLE_PARIS_FILMS_BB, PICKLE_PARIS_FILMS_BB)
log_msg('Done saving poster image links into BlackBlaze')


31-12-2025 17:26:40 Loading films
31-12-2025 17:26:40 Getting bucket...
31-12-2025 17:26:42 Done
31-12-2025 17:26:42 Downloading file from BB:berger_films.pickle
31-12-2025 17:26:43 Save file berger_films.pickle
31-12-2025 17:26:43 Done
31-12-2025 17:26:43 Downloaded 'berger_films.pickle' to 'berger_films.pickle'
31-12-2025 17:26:43 Getting bucket...
31-12-2025 17:26:45 Done
31-12-2025 17:26:45 Downloading file from BB:berger_films_failed.pickle
31-12-2025 17:26:45 Save file berger_films_failed.pickle
31-12-2025 17:26:45 Done
31-12-2025 17:26:45 Downloaded 'berger_films_failed.pickle' to 'berger_films_failed.pickle'
31-12-2025 17:26:45 Adding film posters
31-12-2025 17:27:50 Saving poster links
31-12-2025 17:27:53 Done saving poster image links into BlackBlaze


In [ ]:
############################################################
# GET TMDB MOVE ID

log_msg('Loading films')
x_perplexity, x_failed = get_existing_films()
if 'TMDB_ID' not in x_perplexity.columns:
    x_perplexity['TMDB_ID'] = None

# Loop through all films
for f in x_perplexity.index:
    if pd.isnull(x_perplexity.loc[f,'TMDB_ID']):
        if isinstance(f, str):
            log_msg('Fetching TMBD ID for '+f)
            movie_id = lib_tmdb.tmdb_get_movie_id(
                        title=html.unescape(f),
                        director=html.unescape(x_perplexity.loc[f,'Director'])
                    )
            x_perplexity.loc[f,'TMDB_ID'] = movie_id

# Save result
log_msg('Saving TMDB IDs')
x_perplexity.to_pickle(PICKLE_PARIS_FILMS_BB)
push_b2_file('bergershops', PICKLE_PARIS_FILMS_BB, PICKLE_PARIS_FILMS_BB)
log_msg('Done saving TMDB IDs into BlackBlaze')


31-12-2025 19:49:45 Loading films
31-12-2025 19:49:45 Getting bucket...
31-12-2025 19:49:46 Done
31-12-2025 19:49:46 Downloading file from BB:berger_films.pickle
31-12-2025 19:49:47 Save file berger_films.pickle
31-12-2025 19:49:47 Done
31-12-2025 19:49:47 Downloaded 'berger_films.pickle' to 'berger_films.pickle'
31-12-2025 19:49:47 Getting bucket...
31-12-2025 19:49:49 Done
31-12-2025 19:49:49 Downloading file from BB:berger_films_failed.pickle
31-12-2025 19:49:49 Save file berger_films_failed.pickle
31-12-2025 19:49:49 Done
31-12-2025 19:49:49 Downloaded 'berger_films_failed.pickle' to 'berger_films_failed.pickle'
31-12-2025 19:49:49 Fetching TMBD ID for First Window
31-12-2025 19:49:50 Fetching TMBD ID for Golmanov Strah Od Jedanaesterca
31-12-2025 19:49:50 Fetching TMBD ID for Grand Manuever
31-12-2025 19:49:50 Fetching TMBD ID for Les Ascensions de Werner Herzog: La Soufriere & Gasherbrum
31-12-2025 19:49:51 Fetching TMBD ID for La Filmothèque du Quartier Latin ・ Cinéma parisien d

In [52]:
import importlib
import lib_tmdb

importlib.reload(lib_tmdb)

movie_id = 62  # from your stored data

#images  = lib_tmdb.tmdb_get_movie_images(movie_id)
#videos  = lib_tmdb.tmdb_get_movie_videos(movie_id)

#poster_url = lib_tmdb.tmdb_poster_url_from_id(movie_id, size="w342")
#print(poster_url)
#print(details)
#print(credits)
#print(images)
#poster_url = lib_tmdb.pick_best_poster(images)
# 
#print(poster_url)
#print(videos)
s = lib_tmdb.tmdb_get_trailer_url(movie_id)
print(s)

https://www.youtube.com/watch?v=kR2r-A9H3Kg


In [ ]:
############################################################
# GET TMDB MOVE ID

import lib_tmdb
importlib.reload(lib_tmdb)

log_msg('Loading films')
x_perplexity, x_failed = get_existing_films()
for f in ['TMDB_Cast','TMDB_Country','TMDB_Genre','TMDB_Poster','TMDB_Popularity','TMDB_Release_Date','TMDB_Runtime','TMDB_Synopsis','TMDB_Poster','TMDB_Trailer']:
    if f not in x_perplexity.columns:
        x_perplexity[f] = None

# Loop through all films
log_msg('Extracting film details from TMDB')
for f in x_perplexity.index:
    if True or pd.isnull(x_perplexity.loc[f,'TMDB_Cast']):
        if isinstance(f, str):
            log_msg('Fetching TMBD details for '+f)
            # Download TMDB details
            movie_id = x_perplexity.loc[f,'TMDB_ID']
            try:
                details = lib_tmdb.tmdb_get_movie_details(movie_id)
                credits = lib_tmdb.tmdb_get_movie_credits(movie_id)
                images = lib_tmdb.tmdb_get_movie_images(movie_id)
                film_trailer = lib_tmdb.tmdb_get_trailer_url(movie_id)

                # Extract values
                if isinstance(details['origin_country'], list):
                    film_country = ' | '.join(details['origin_country'])
                else:
                    film_country = details['origin_country']
                film_synopsis = details['overview']
                film_popularity = details['popularity']
                film_release_date = details['release_date']
                film_runtime = details['runtime']
                film_genre = ' | '.join([ k['name'] for k in details['genres']])
                film_poster = lib_tmdb.pick_best_poster(images)
                film_poster = lib_tmdb.tmdb_poster_url_from_poster_dict(film_poster, size="w342")
                film_cast = ''
                for actor in credits['cast'][:5]:
                    if film_cast != '':
                        film_cast += '\n'
                    film_cast += actor['name'] + ' (' + actor['character'] + ')'
            except Exception as e:
                log_msg('Failed to get TMDB details for ' + f + ' | ' + str(e))
                film_country = ''
                film_synopsis = ''
                film_popularity = ''
                film_release_date = ''
                film_runtime = ''
                film_genre = ''
                film_poster = ''
                film_cast = ''
                film_trailer = ''

            # Store
            x_perplexity.loc[f,'TMDB_Cast'] = film_cast
            x_perplexity.loc[f,'TMDB_Country'] = film_country
            x_perplexity.loc[f,'TMDB_Genre'] = film_genre
            x_perplexity.loc[f,'TMDB_Poster'] = film_poster
            x_perplexity.loc[f,'TMDB_Popularity'] = film_popularity
            x_perplexity.loc[f,'TMDB_Release_Date'] = film_release_date
            x_perplexity.loc[f,'TMDB_Runtime'] = film_runtime
            x_perplexity.loc[f,'TMDB_Synopsis'] = film_synopsis
            x_perplexity.loc[f,'TMDB_Trailer'] = film_trailer



# Save result
log_msg('Saving TMDB details')
x_perplexity.to_pickle(PICKLE_PARIS_FILMS_BB)
push_b2_file('bergershops', PICKLE_PARIS_FILMS_BB, PICKLE_PARIS_FILMS_BB)
log_msg('Done saving TMDB details into BlackBlaze')


31-12-2025 23:07:18 Loading films
31-12-2025 23:07:18 Getting bucket...
31-12-2025 23:07:21 Done
31-12-2025 23:07:21 Downloading file from BB:berger_films.pickle
31-12-2025 23:07:22 Save file berger_films.pickle
31-12-2025 23:07:22 Done
31-12-2025 23:07:22 Downloaded 'berger_films.pickle' to 'berger_films.pickle'
31-12-2025 23:07:22 Getting bucket...
31-12-2025 23:07:24 Done
31-12-2025 23:07:24 Downloading file from BB:berger_films_failed.pickle
31-12-2025 23:07:25 Save file berger_films_failed.pickle
31-12-2025 23:07:25 Done
31-12-2025 23:07:25 Downloaded 'berger_films_failed.pickle' to 'berger_films_failed.pickle'
31-12-2025 23:07:25 Extracting film details from TMDB
31-12-2025 23:07:25 Fetching TMBD details for 2001: A Space Odyssey
31-12-2025 23:07:27 Fetching TMBD details for Aftersun
31-12-2025 23:07:29 Fetching TMBD details for Amélie
31-12-2025 23:07:31 Fetching TMBD details for Anastasia
31-12-2025 23:07:33 Fetching TMBD details for Angel's Egg
31-12-2025 23:07:34 Fetching TMB

In [58]:
x_perplexity.to_clipboard()

In [ ]:
table_filled = table_films.join(x_perplexity, on='Title', how='left')

In [25]:
from html import escape
header_color = "#C0716D"
divider_color = "#E0E0E0"
table_df = table_filled.reset_index()
if 'Title' not in table_df.columns:
    table_df = table_df.rename(columns={'index': 'Title'})
table_df = table_df.fillna('')
def _as_text(value):
    value = '' if value is None else value
    return str(value).strip()
def _format_multiline(value):
    return escape(_as_text(value)).replace('\n', '<br>')
table_style = f"""
<style>
  .film-table-container {{
    width: 100%;
    font-family: 'Helvetica Neue', Arial, sans-serif;
    font-size: 14px;
    color: #000000;
  }}
  .film-table {{
    width: 100%;
    border-collapse: collapse;
    table-layout: fixed;
  }}
  .film-table thead th {{
    text-align: left;
    color: {header_color};
    border-bottom: 1px solid {header_color};
    font-weight: 600;
    padding: 8px 6px;
    white-space: nowrap;
  }}
  .film-table tbody td {{
    color: #000000;
    border-bottom: 1px solid {divider_color};
    padding: 10px 6px;
    vertical-align: top;
    word-break: break-word;
  }}
  .film-table tbody tr:last-child td {{
    border-bottom: none;
  }}
  .film-eye-btn,
  .film-youtube-btn {{
    background: none;
    border: none;
    cursor: pointer;
    font-size: 1rem;
    color: {header_color};
  }}
  .film-eye-btn:focus,
  .film-youtube-btn:focus {{
    outline: 2px solid {header_color};
    outline-offset: 2px;
  }}
  .film-youtube-btn.disabled {{
    cursor: default;
    color: #bbbbbb;
  }}
  .col-title {{ width: 18.9%; min-width: 220px; }}
  .col-eye {{ width: 3.3%; text-align: center; }}
  .col-youtube {{ width: 3.3%; text-align: center; }}
  .col-director {{ width: 12%; min-width: 120px; }}
  .col-year {{ width: 6.9%; min-width: 65px; }}
  .col-genre {{ width: 8.6%; min-width: 95px; }}
  .col-country {{ width: 8.6%; min-width: 90px; }}
  .col-cinemas {{ width: 15.4%; min-width: 210px; }}
  .col-schedule {{ width: 23.2%; min-width: 230px; }}
  .synopsis-modal {{
    display: none;
    position: fixed;
    z-index: 9999;
    left: 0;
    top: 0;
    width: 100%;
    height: 100%;
    background: rgba(0,0,0,0.45);
  }}
  .synopsis-modal-content {{
    background: #ffffff;
    margin: 10% auto;
    padding: 20px 24px;
    border-radius: 8px;
    max-width: 600px;
    box-shadow: 0 8px 24px rgba(0,0,0,0.2);
  }}
  .synopsis-modal-content h3 {{
    margin-top: 0;
    color: {header_color};
  }}
  .close-synopsis {{
    float: right;
    cursor: pointer;
    font-weight: bold;
    font-size: 18px;
    color: {header_color};
  }}
  @media (max-width: 900px) {{
    .film-table {{ table-layout: auto; }}
    .film-table thead th, .film-table tbody td {{ padding: 8px 4px; }}
  }}
</style>
"""
table_script = """
<script>
function openSynopsis(id) {
  var el = document.getElementById(id);
  if (el) {
    el.style.display = 'block';
  }
}
function closeSynopsis(event, id) {
  var el = document.getElementById(id);
  if (!el) { return; }
  if (!event || event.target.classList.contains('synopsis-modal') || event.target.classList.contains('close-synopsis')) {
    el.style.display = 'none';
  }
}
function openYouTube(url) {
  if (url) {
    window.open(url, '_blank', 'noopener');
  }
}
</script>
"""
headers = [
    "Title","","","Director","Year","Genre","Country","Cinemas","Schedule"
 ]
column_classes = ['col-title','col-eye','col-youtube','col-director','col-year','col-genre','col-country','col-cinemas','col-schedule']
table_parts = [table_style, table_script, '<div class="film-table-container">', '<table class="film-table">']
table_parts.append('<colgroup>')
for cls in column_classes:
    table_parts.append(f"<col class='{cls}'>")
table_parts.append('</colgroup>')
table_parts.append('<thead><tr>')
for head in headers:
    table_parts.append(f"<th>{head}</th>")
table_parts.extend(['</tr></thead>', '<tbody>'])
modals = []
for idx, row in table_df.iterrows():
    title = _as_text(row.get('Title')) or 'Untitled'
    link = _as_text(row.get('Film link'))
    title_cell = escape(title) if not link else f"<a href='{escape(link, quote=True)}' target='_blank' rel='noopener noreferrer'>{escape(title)}</a>"
    synopsis_html = _format_multiline(row.get('Synopsis') or 'Synopsis unavailable.')
    modal_id = f"synopsis-{idx}"
    eye_cell = f"<button class='film-eye-btn' type='button' onclick=\"openSynopsis('{modal_id}')\" aria-label='Synopsis'>&#128065;</button>"
    youtube_link = _as_text(row.get('YouTube'))
    if youtube_link:
        yt_button = f"<button class='film-youtube-btn' type='button' onclick=\"openYouTube('{escape(youtube_link, quote=True)}')\" aria-label='Play trailer'>&#9654;</button>"
    else:
        yt_button = "<button class='film-youtube-btn disabled' type='button' disabled aria-label='No trailer'>&#9654;</button>"
    row_cells = [
        title_cell,
        eye_cell,
        yt_button,
        _format_multiline(row.get('Director')),
        escape(_as_text(row.get('Year'))),
        escape(_as_text(row.get('Genre'))),
        escape(_as_text(row.get('Country'))),
        _format_multiline(row.get('Cinemas')),
        _format_multiline(row.get('Schedule'))
    ]
    table_parts.append('<tr>')
    for cell, cls in zip(row_cells, column_classes):
        class_attr = f" class='{cls}'" if cls else ''
        table_parts.append(f"<td{class_attr}>{cell}</td>")
    table_parts.append('</tr>')
    modal_html = f"""
<div id="{modal_id}" class="synopsis-modal" onclick="closeSynopsis(event, '{modal_id}')">
  <div class="synopsis-modal-content">
    <span class="close-synopsis" onclick="closeSynopsis(event, '{modal_id}')">&times;</span>
    <h3>{escape(title)}</h3>
    <p>{synopsis_html}</p>
  </div>
</div>
"""
    modals.append(modal_html)
table_parts.extend(['</tbody>', '</table>'])
table_parts.extend(modals)
table_parts.append('</div>')
table_HTML = ''.join(table_parts)
with open('/Users/etiennecomon/Downloads/out6.html','wt') as f:
    f.write(table_HTML)
print('done with file')

done with file


In [23]:
import pandas as pd
import html

def safe(value):
    """Convert value to escaped string, treating NaN as empty."""
    if pd.isna(value):
        return ""
    return html.escape(str(value))

table_parts = []

# CSS styles
table_styles = """
<style>
.film-table-container {
    font-family: Arial, sans-serif;
    font-size: 14px;
}

.film-table {
    width: 100%;
    border-collapse: collapse;
    border: none;
}

.film-table thead th {
    color: #C0716D;
    font-weight: 600;
    text-align: left;
    padding: 8px 10px;
    border-bottom: 2px solid #C0716D;
}

.film-table tbody td {
    color: #000000;
    padding: 8px 10px;
    border: none;
}

.film-table tbody tr {
    border-bottom: 1px solid #e0e0e0; /* light grey horizontal separators */
}

.film-table tbody tr:last-child {
    border-bottom: none;
}

.film-table tbody tr:hover {
    background-color: #faf7f7;
}

.film-table a {
    color: inherit;
    text-decoration: none;
    border-bottom: 1px dotted #C0716D;
}

.film-table a:hover {
    text-decoration: underline;
}

.eye-cell {
    width: 40px;
    text-align: center;
}

.eye-button {
    background: none;
    border: none;
    cursor: pointer;
    font-size: 16px;
    line-height: 1;
}

.eye-button:hover {
    transform: scale(1.1);
}

/* Modal styles */
.synopsis-modal {
    display: none;
    position: fixed;
    z-index: 9999;
    left: 0;
    top: 0;
    width: 100%;
    height: 100%;
    overflow: auto;
    background-color: rgba(0,0,0,0.4);
}

.synopsis-modal-content {
    background-color: #ffffff;
    margin: 10% auto;
    padding: 20px 24px;
    border-radius: 8px;
    max-width: 600px;
    box-shadow: 0 4px 16px rgba(0,0,0,0.2);
}

.synopsis-modal-content h3 {
    margin-top: 0;
    margin-bottom: 12px;
    color: #C0716D;
}

.synopsis-modal-content p {
    margin: 0;
    white-space: normal;
}

.close-synopsis {
    float: right;
    font-size: 20px;
    font-weight: bold;
    cursor: pointer;
    margin-left: 12px;
}
</style>
"""

# JS for opening/closing synopsis modals
scripts = """
<script>
function openSynopsis(id) {
    var el = document.getElementById(id);
    if (el) {
        el.style.display = 'block';
    }
}

function closeSynopsis(event, id) {
    var el = document.getElementById(id);
    if (!el) return;
    if (!event || event.target.classList.contains('synopsis-modal') ||
        event.target.classList.contains('close-synopsis')) {
        el.style.display = 'none';
    }
}
</script>
"""

table_parts.append('<div class="film-table-container">')
table_parts.append(table_styles)
table_parts.append('<table class="film-table">')

# Header
table_parts.append("""
<thead>
  <tr>
    <th>Title</th>
    <th></th>
    <th>Director</th>
    <th>Year</th>
    <th>Genre</th>
    <th>Country</th>
    <th>Cinemas</th>
    <th>Schedule</th>
  </tr>
</thead>
<tbody>
""")

modals = []

# Rows
for idx, row in table_filled.iterrows():
    title = safe(row.get("Title"))
    link = safe(row.get("Film link"))
    director = safe(row.get("Director"))
    year = safe(row.get("Year"))
    genre = safe(row.get("Genre"))
    country = safe(row.get("Country"))
    cinemas = safe(row.get("Cinemas"))
    schedule = safe(row.get("Schedule"))
    synopsis_html = safe(row.get("Synopsis")).replace("\n", "<br>")
    modal_id = f"synopsis-{idx}"

    row_html = f"""
  <tr>
    <td class="title-cell">
      <a href="{link}" target="_blank" rel="noopener noreferrer">{title}</a>
    </td>
    <td class="eye-cell">
      <button type="button" class="eye-button" onclick="openSynopsis('{modal_id}')">&#128065;</button>
    </td>
    <td>{director}</td>
    <td>{year}</td>
    <td>{genre}</td>
    <td>{country}</td>
    <td>{cinemas}</td>
    <td>{schedule}</td>
  </tr>
"""
    table_parts.append(row_html)

    modal_html = f"""
<div id="{modal_id}" class="synopsis-modal" onclick="closeSynopsis(event, '{modal_id}')">
  <div class="synopsis-modal-content">
    <span class="close-synopsis" onclick="closeSynopsis(event, '{modal_id}')">&times;</span>
    <h3>{title}</h3>
    <p>{synopsis_html}</p>
  </div>
</div>
"""
    modals.append(modal_html)

table_parts.append("</tbody></table>")
table_parts.extend(modals)
table_parts.append(scripts)
table_parts.append("</div>")

table_HTML = "\n".join(table_parts)
with open('/Users/etiennecomon/Downloads/out4.html','wt') as f:
    f.write(table_HTML)